# Input and output formats

`sfsutils` reads variants from a VCF file, a VCF-Zarr store, or a tskit tree sequence through a single streamed site interface, so the same analysis code works for any input format. Writing follows the output file's extension.

The dataset here is one synthetic ARG, provided as a tree sequence together with the VCF and VCF-Zarr store converted from it. Since all three encode the same genotypes, they yield the same spectrum.

## Three input forms

~~~~~~{tab-set} 
:sync-group: language
:class: code-tabs

~~~~~{tab-item} {fab}`python` Python
:sync: python

```python
trees = "resources/msprime/two_epoch.trees"  # tskit tree sequence (the ARG)
vcf = "resources/msprime/two_epoch.vcf"  # VCF written from it
vcz = "resources/msprime/two_epoch.vcz"  # VCF-Zarr store converted from the VCF
```

~~~~~

~~~~~{tab-item} {fab}`r-project` R
:sync: r

```r
trees <- "resources/msprime/two_epoch.trees"  # tskit tree sequence (the ARG)
vcf <- "resources/msprime/two_epoch.vcf"  # VCF written from it
vcz <- "resources/msprime/two_epoch.vcz"  # VCF-Zarr store converted from the VCF
```

~~~~~

~~~~~~


## Reading

{class}`~sfsutils.parser.Parser` accepts any of the three as its `source` argument and infers the backend from the source. Reading a VCF-Zarr store needs the optional `zarr` package, a tree sequence the optional `tskit` package.

~~~~~~{tab-set} 
:sync-group: language
:class: code-tabs

~~~~~{tab-item} {fab}`python` Python
:sync: python

```python
import numpy as np
import sfsutils as su

sfs_trees = su.Parser(source=trees, n=10, skip_non_polarized=False).parse()
sfs_vcf = su.Parser(source=vcf, n=10, skip_non_polarized=False).parse()
sfs_vcz = su.Parser(source=vcz, n=10, skip_non_polarized=False).parse()
```

```{glue} python-s2-0-0
```

```python
# the three spectra are identical
np.array_equal(sfs_trees.all.data, sfs_vcf.all.data) and np.array_equal(sfs_vcf.all.data, sfs_vcz.all.data)
```

```{glue} python-s2-1-0
```

```python
sfs_vcf.all.plot();
```

```{glue} python-s2-2-0
```

~~~~~

~~~~~{tab-item} {fab}`r-project` R
:sync: r

```r
library(sfsutils)
su <- load_sfsutils()

sfs_trees <- su$Parser(source = trees, n = 10, skip_non_polarized = FALSE)$parse()
sfs_vcf <- su$Parser(source = vcf, n = 10, skip_non_polarized = FALSE)$parse()
sfs_vcz <- su$Parser(source = vcz, n = 10, skip_non_polarized = FALSE)$parse()
```

```{glue} r-s2-0-0
```

```r
# the three spectra are identical
identical(sfs_trees$all$to_list(), sfs_vcf$all$to_list()) && identical(sfs_vcf$all$to_list(), sfs_vcz$all$to_list())
```

```{glue} r-s2-1-0
```

```r
p <- sfs_vcf$all$plot()
```

```{glue} r-s2-2-0
```

~~~~~

~~~~~~


## Writing

{class}`~sfsutils.filtration.Filterer` and {class}`~sfsutils.annotation.Annotator` pick the writer from the output file's extension: `.vcf`/`.vcf.gz` for a VCF, `.vcz`/`.zarr` for a VCF-Zarr store, and `.trees` for a tree sequence. A tree sequence can only be written from a tree-sequence input: filtering removes the discarded sites with `delete_sites`, leaving the genealogy intact. A genealogy cannot be reconstructed from genotype data, so writing a `.trees` from a VCF or VCF-Zarr store is rejected.

~~~~~~{tab-set} 
:sync-group: language
:class: code-tabs

~~~~~{tab-item} {fab}`python` Python
:sync: python

```python
import os
import tempfile

out = tempfile.mkdtemp()

# a VCF-Zarr store can be written from any input
su.Filterer(source=vcf, output=os.path.join(out, "snps.vcz"), filtrations=[su.SNPFiltration()]).filter()

# a VCF is written from a VCF input
su.Filterer(source=vcf, output=os.path.join(out, "snps.vcf"), filtrations=[su.SNPFiltration()]).filter()

# a tree sequence is written from a tree-sequence input
su.Filterer(source=trees, output=os.path.join(out, "snps.trees"), filtrations=[su.SNPFiltration()]).filter()
```

```{glue} python-s3-0-0
```

~~~~~

~~~~~{tab-item} {fab}`r-project` R
:sync: r

```r
out <- tempfile()
dir.create(out)

# a VCF-Zarr store can be written from any input
su$Filterer(source = vcf, output = file.path(out, "snps.vcz"), filtrations = list(su$SNPFiltration()))$filter()

# a VCF is written from a VCF input
su$Filterer(source = vcf, output = file.path(out, "snps.vcf"), filtrations = list(su$SNPFiltration()))$filter()

# a tree sequence is written from a tree-sequence input
su$Filterer(source = trees, output = file.path(out, "snps.trees"), filtrations = list(su$SNPFiltration()))$filter()
```

```{glue} r-s3-0-0
```

~~~~~

~~~~~~


The store and tree sequence written above parse back to the same spectrum as the VCF output.

~~~~~~{tab-set} 
:sync-group: language
:class: code-tabs

~~~~~{tab-item} {fab}`python` Python
:sync: python

```python
back = [
    su.Parser(source=os.path.join(out, f), n=10, skip_non_polarized=False).parse().all.data
    for f in ["snps.vcf", "snps.vcz", "snps.trees"]
]

all(np.array_equal(b, back[0]) for b in back)
```

```{glue} python-s4-0-0
```

```{glue} python-s4-0-1
```

~~~~~

~~~~~{tab-item} {fab}`r-project` R
:sync: r

```r
back <- lapply(c("snps.vcf", "snps.vcz", "snps.trees"), function(f) {
  su$Parser(source = file.path(out, f), n = 10, skip_non_polarized = FALSE)$parse()$all$to_list()
})

all(vapply(back, identical, logical(1), back[[1]]))
```

```{glue} r-s4-0-0
```

```{glue} r-s4-0-1
```

~~~~~

~~~~~~
